In [12]:
from astropy.cosmology import Planck18
%env XLA_PYTHON_CLIENT_ALLOCATOR=platform
import astropy.units as u
import warnings
warnings.filterwarnings("ignore", "Wswiglal-redir-stdio")
from glob import glob
import numpy as np
import sys
sys.path.append('../src/')
import jax.numpy as jnp
import matplotlib.pyplot as plt
from gwosc.api import fetch_event_json
import re
import os
import jax
import h5py
import pandas as pd
jax.local_devices()
jax.config.update("jax_enable_x64", True)


env: XLA_PYTHON_CLIENT_ALLOCATOR=platform


In [13]:
#data_paths = ['/mnt/home/ccalvk/ceph/GWTC-4/IGWN-GWTC4p0-*-combined_PEDataRelease.hdf5',]
#data_paths=['/mnt/home/ccalvk/ceph/GWTC-3/IGWN-GWTC3p0-v2-GW*_PEDataRelease_mixed_nocosmo.h5']
data_paths=[
    '/mnt/home/ccalvk/ceph/GWTC-2.1/IGWN-GWTC2p1-v2-GW*_PEDataRelease_mixed_nocosmo.h5',
    #'/mnt/home/ccalvk/ceph/GWTC-4/IGWN-GWTC4p0-*-combined_PEDataRelease.hdf5',
    '/mnt/home/misi/ceph/rp.04/catalogs/GWTC-4p1/GWTC4p1-Stable_Release-3/18965dda8_5/bbh_only/IGWN-GWTC4p1-18965dda8_5-GW*-combined_PEDataRelease.hdf5',
    '/mnt/home/ccalvk/ceph/GWTC-3/IGWN-GWTC3p0-v2-GW*_PEDataRelease_mixed_nocosmo.h5',
    '/mnt/home/misi/ceph/rp.04/catalogs/GWTC-5/GWTC5-Stable_Release-8/4c51f4201_25/bbh_only/IGWN-GWTC5p0-*-combined_PEDataRelease.hdf5'
    ]
files = []
for path in data_paths:
    files += glob(path)

print(len(files))

280


In [14]:
import requests

r = requests.get("https://gwosc.org/eventapi/csv/allevents/", timeout=30)
print(r.status_code)
print(r.text[:500])

200
id,commonName,version,catalog.shortName,GPS,reference,jsonurl,mass_1_source,mass_1_source_lower,mass_1_source_upper,mass_2_source,mass_2_source_lower,mass_2_source_upper,network_matched_filter_snr,network_matched_filter_snr_lower,network_matched_filter_snr_upper,luminosity_distance,luminosity_distance_lower,luminosity_distance_upper,chi_eff,chi_eff_lower,chi_eff_upper,total_mass_source,total_mass_source_lower,total_mass_source_upper,chirp_mass_source,chirp_mass_source_lower,chirp_mass_source_upp


In [ ]:
import os
import re
for file in files:
        
    #filename = os.path.basename(file)
    match = re.search(r"(GW\d{6}(?:_\d{6})?)", os.path.basename(file))
    if not match:
        print('issue')
    event = match.group(1)
    parts = re.split("_|-", filename)
    data_release=parts[1]
    print(event)
    #if data_release=='GWTC2p1':
    #    data_release='GWTC2.1'


GW190513_205428
GW190426_190642
GW190725_174728
GW190720_000836
GW170823_131358
GW190521_074359
GW190828_065509
GW190408_181802
GW190828_063405
GW190425_081805
GW190403_051519
GW190805_211137
GW170729_185629
GW190916_200658
GW190925_232845
GW190814_211039
GW190602_175927
GW190412_053044
GW190917_114630
GW190421_213856
GW190926_050336
GW190910_112807
GW190413_052954
GW190519_153544
GW190503_185404
GW170818_022509
GW190707_093326
GW190630_185205
GW190727_060333
GW190521_030229
GW190620_030421
GW190413_134308
GW190527_092055
GW190719_215514
GW190728_064510
GW151012_095443
GW190514_065416
GW170608_020116
GW190929_012149
GW150914_095045
GW190701_203306
GW190930_133541
GW190706_222641
GW170814_103043
GW170104_101158
GW190708_232457
GW190915_235702
GW190731_140936
GW190517_055101
GW190924_021846
GW151226_033853
GW190512_180714
GW170809_082821
GW190803_022701
GW231118_090602
GW231206_233134
GW231028_153006
GW230731_215307
GW231008_142521
GW230712_090405
GW230729_082317
GW230723_101834
GW240109

In [16]:
import os
import re
import csv
import io
import requests
 
_FAR_CACHE = None
ALLEVENTS_CSV_URL = "https://gwosc.org/eventapi/csv/allevents/"
 
 
def _load_far_table(timeout=60):
    resp = requests.get(ALLEVENTS_CSV_URL, timeout=timeout)
    resp.raise_for_status()
 
    reader = csv.DictReader(io.StringIO(resp.text))
    rows = list(reader)
 
    print(f"[debug] downloaded {len(rows)} rows")
    print(f"[debug] columns: {reader.fieldnames}")
 
    # Only trust official LVK catalog releases. Independent/community
    # reanalyses (e.g. "IAS-O3a", "OGC-...") can report very different
    # FAR values for the same event and must not silently overwrite
    # the official value just because they appear later in the file.
    OFFICIAL_CATALOG_PREFIXES = ("GWTC-1", "GWTC-2", "GWTC-3", "GWTC-4", "GWTC-5.0")
 
    cache = {}
    cache_version = {}  # name -> version number already stored
    skipped_catalogs = set()
    for row in rows:
        catalog = row.get("catalog.shortName", "")
        if not catalog.startswith(OFFICIAL_CATALOG_PREFIXES):
            skipped_catalogs.add(catalog)
            continue
 
        far_str = row.get("far", "")
        if not far_str:
            continue
        try:
            far = float(far_str)
        except ValueError:
            continue
 
        common = row.get("commonName", "").strip()
        if not common:
            continue
        name = common if common.startswith("GW") else f"GW{common}"
 
        try:
            version = int(row.get("version", 0))
        except ValueError:
            version = 0
 
        if name not in cache_version or version > cache_version[name]:
            cache[name] = far
            cache_version[name] = version
 
    print(f"[debug] ignored non-official catalogs: {sorted(skipped_catalogs)}")
 
    print(f"[debug] {len(cache)} events with a FAR value")
    print(f"[debug] sample keys: {list(cache.items())[:5]}")
    return cache
 
 
def get_event_far(file):
    global _FAR_CACHE
 
    match = re.search(r"(GW\d{6}(?:_\d{6})?)", os.path.basename(file))
    if not match:
        return None, None
 
    event = match.group(1)
    base_event = event.split("_")[0]
 
    if _FAR_CACHE is None:
        try:
            _FAR_CACHE = _load_far_table()
        except Exception as e:
            print(f"Failed to load FAR table: {e}")
            _FAR_CACHE = {}
 
    if event in _FAR_CACHE:
        return _FAR_CACHE[event], event
    if base_event in _FAR_CACHE:
        return _FAR_CACHE[base_event], event
 
    print(f"No FAR found for {event}")
    return None, event

In [17]:
fars = []
names = []
for file in files:
    far, event = get_event_far(file)
    if far is None:
        continue
    if far < 1:
        fars.append(far)
        names.append(event)

[debug] downloaded 671 rows
[debug] columns: ['id', 'commonName', 'version', 'catalog.shortName', 'GPS', 'reference', 'jsonurl', 'mass_1_source', 'mass_1_source_lower', 'mass_1_source_upper', 'mass_2_source', 'mass_2_source_lower', 'mass_2_source_upper', 'network_matched_filter_snr', 'network_matched_filter_snr_lower', 'network_matched_filter_snr_upper', 'luminosity_distance', 'luminosity_distance_lower', 'luminosity_distance_upper', 'chi_eff', 'chi_eff_lower', 'chi_eff_upper', 'total_mass_source', 'total_mass_source_lower', 'total_mass_source_upper', 'chirp_mass_source', 'chirp_mass_source_lower', 'chirp_mass_source_upper', 'chirp_mass', 'chirp_mass_lower', 'chirp_mass_upper', 'redshift', 'redshift_lower', 'redshift_upper', 'far', 'far_lower', 'far_upper', 'p_astro', 'p_astro_lower', 'p_astro_upper', 'final_mass_source', 'final_mass_source_lower', 'final_mass_source_upper']
[debug] ignored non-official catalogs: ['IAS-O3a', 'Initial_LIGO_Virgo', 'O1_O2-Preliminary', 'O3_Discovery_Pape

KeyboardInterrupt: 

In [ ]:
resp = requests.get("https://gwosc.org/eventapi/csv/allevents/", timeout=60)
resp.raise_for_status()
 
reader = csv.DictReader(io.StringIO(resp.text))
for row in reader:
    if "190916" in row.get("commonName", "") or "190916" in row.get("id", ""):
        print(row["id"], row["commonName"], row["version"], row["catalog.shortName"], "far=", row["far"])

GW190916_200658-v1 GW190916_200658 1 GWTC-2.1-confident far= 4.7
GW190916_200658-v2 GW190916_200658 2 IAS-O3a far= 0.048


In [ ]:
event_far, name = get_event_far('/mnt/home/ccalvk/ceph/GWTC-2.1/IGWN-GWTC2p1-v2-_GW190513_205428_PEDataRelease_mixed_nocosmo.h5')
#event_far, name = get_event_far('/mnt/home/ccalvk/ceph/GWTC-4/IGWN-GWTC4p0-1a206db3d_721-GW231026_130704-combined_PEDataRelease.hdf5')

print(f"{name}: far={event_far!r}")

GW190513_205428: far=1.3e-05


In [ ]:
far_threshold = 1
mass_sel = 2.5

include = []
fars = []

special_events = {
    'GW150914_095045',
    'GW200129_065458',
    'GW190521_074359',
    'GW190521_030229',
}

# Waveform preferences, in order
normal_preference = [
    'C00:NRSur7dq4',
    'C01:Mixed',
    'C00:Mixed',
    'C00:IMRPhenomXPHM-SpinTaylor',
    'C01:IMRPhenomXPHM-SpinTaylor',
    'C01:IMRPhenomXPHM',
    'C01:IMRPhenomPv2_NRTidal:HighSpin',
]

special_preference = [
    'C00:NRSur7dq4',
    'C01:IMRPhenomXPHM',
    'C00:IMRPhenomXPHM-SpinTaylor',
    'C01:IMRPhenomXPHM-SpinTaylor',
    'C01:IMRPhenomPv2_NRTidal:HighSpin',
    'C01:Mixed',
    'C00:Mixed',
]

for file in files:
    event_far, name = get_event_far(file)
    fars.append(event_far)
    print(name)
    with h5py.File(file, 'r') as f:

        # Choose the appropriate waveform preference
        preference = (
            special_preference
            if name in special_events
            else normal_preference
        )

        # Pick the first available waveform
        for waveform in preference:
            if waveform in f:
                samples = np.array(f[waveform]['posterior_samples'])
                print(waveform)
                break
        else:
            print(f"Available keys in file {name}: {list(f.keys())}")
            continue

    # Extract posterior samples
    zs = samples['redshift'][()]
    m1_det = samples['mass_1'][()]
    qs = samples['mass_ratio'][()]

    # Convert detector-frame m2 to source-frame m2
    m2s_det = m1_det * qs
    m2s_src = m2s_det / (1 + zs)
    m2s_med = np.median(m2s_src)

    # Handle missing FAR
    if event_far is None:
        print('far None for', name)
        event_far = 100

    # Selection
    if event_far < far_threshold and m2s_med > mass_sel:
        include.append(name)
# then remove GW190814_211039 by hand (on lower mass cutoff)

GW190513_205428
C01:Mixed
GW190426_190642
C01:Mixed
GW190725_174728
C01:Mixed
GW190720_000836
C01:Mixed
GW170823_131358
C01:Mixed
GW190521_074359
C01:IMRPhenomXPHM
GW190828_065509
C01:Mixed
GW190408_181802
C01:Mixed
GW190828_063405
C01:Mixed
GW190425_081805
C01:IMRPhenomPv2_NRTidal:HighSpin
GW190403_051519
C01:Mixed
GW190805_211137
C01:Mixed
GW170729_185629
C01:Mixed
GW190916_200658
C01:Mixed
GW190925_232845
C01:Mixed
GW190814_211039
C01:Mixed
GW190602_175927
C01:Mixed
GW190412_053044
C01:Mixed
GW190917_114630
C01:Mixed
GW190421_213856
C01:Mixed
GW190926_050336
C01:Mixed
GW190910_112807
C01:Mixed
GW190413_052954
C01:Mixed
GW190519_153544
C01:Mixed
GW190503_185404
C01:Mixed
GW170818_022509
C01:Mixed
GW190707_093326
C01:Mixed
GW190630_185205
C01:Mixed
GW190727_060333
C01:Mixed
GW190521_030229
C01:IMRPhenomXPHM
GW190620_030421
C01:Mixed
GW190413_134308
C01:Mixed
GW190527_092055
C01:Mixed
GW190719_215514
C01:Mixed
GW190728_064510
C01:Mixed
GW151012_095443
C01:Mixed
GW190514_065416
C01:Mixe

KeyboardInterrupt: 

In [ ]:
np.

In [ ]:
file1 = open("../runs/INCLUDE_LIST_all.txt", "a")
for name in include:
    file1.write(name + "\n")
file1.close()

In [ ]:
INCLUDE_LIST=[]
with open("../runs/INCLUDE_LIST.txt", "r") as f:
    INCLUDE_LIST = set(line.strip() for line in f if line.strip())
len(INCLUDE_LIST)

256

In [ ]:
INCLUDE_LIST=[]
with open("../runs/INCLUDE_LIST_all.txt", "r") as f:
    for line in f:
        INCLUDE_LIST.append(line.strip())
    #INCLUDE_LIST = set(line.strip() for line in f if line.strip())
    filtered_files = []
for f in files:
    filename = os.path.basename(f)
    parts = re.split("_|-", filename)
    if len(parts) >= 2 and parts[1] != 'GWTC4p0':
        event_name = parts[3] + "_" + parts[4]
        if event_name in INCLUDE_LIST:
            filtered_files.append(f)
    if len(parts) >= 2 and parts[1] == 'GWTC4p0' or parts[1] == 'GWTC5p0':
        event_name = parts[4] + "_" + parts[5]
        if event_name in INCLUDE_LIST:
            filtered_files.append(f)
    print(event_name)

GW190513_205428
GW190426_190642
GW190725_174728
GW190720_000836
GW170823_131358
GW190521_074359
GW190828_065509
GW190408_181802
GW190828_063405
GW190425_081805
GW190403_051519
GW190805_211137
GW170729_185629
GW190916_200658
GW190925_232845
GW190814_211039
GW190602_175927
GW190412_053044
GW190917_114630
GW190421_213856
GW190926_050336
GW190910_112807
GW190413_052954
GW190519_153544
GW190503_185404
GW170818_022509
GW190707_093326
GW190630_185205
GW190727_060333
GW190521_030229
GW190620_030421
GW190413_134308
GW190527_092055
GW190719_215514
GW190728_064510
GW151012_095443
GW190514_065416
GW170608_020116
GW190929_012149
GW150914_095045
GW190701_203306
GW190930_133541
GW190706_222641
GW170814_103043
GW170104_101158
GW190708_232457
GW190915_235702
GW190731_140936
GW190517_055101
GW190924_021846
GW151226_033853
GW190512_180714
GW170809_082821
GW190803_022701
5_GW231118
5_GW231206
5_GW231028
5_GW230731
5_GW231008
5_GW230712
5_GW230729
5_GW230723
5_GW240109
5_GW231114
5_GW230920
5_GW231118
5_GW

In [ ]:
len(INCLUDE_LIST)

259

In [ ]:
unq, unq_idx, unq_cnt = np.unique(INCLUDE_LIST, return_inverse=True, return_counts=True)


In [ ]:
len(unq)

258

In [ ]:
for name in unq:
    print(name)

GW150914_095045
GW151012_095443
GW151226_033853
GW170104_101158
GW170608_020116
GW170729_185629
GW170809_082821
GW170814_103043
GW170818_022509
GW170823_131358
GW190408_181802
GW190412_053044
GW190413_052954
GW190413_134308
GW190421_213856
GW190503_185404
GW190512_180714
GW190513_205428
GW190514_065416
GW190517_055101
GW190519_153544
GW190521_030229
GW190521_074359
GW190527_092055
GW190602_175927
GW190620_030421
GW190630_185205
GW190701_203306
GW190706_222641
GW190707_093326
GW190708_232457
GW190719_215514
GW190720_000836
GW190725_174728
GW190727_060333
GW190728_064510
GW190731_140936
GW190803_022701
GW190805_211137
GW190814_211039
GW190828_063405
GW190828_065509
GW190910_112807
GW190915_235702
GW190916_200658
GW190924_021846
GW190925_232845
GW190926_050336
GW190929_012149
GW190930_133541
GW191103_012549
GW191105_143521
GW191109_010717
GW191127_050227
GW191129_134029
GW191204_171526
GW191215_223052
GW191216_213338
GW191222_033537
GW191230_180458
GW200112_155838
GW200128_022011
GW200129

In [ ]:
cnt_mask = unq_cnt > 1
dup_ids = unq[cnt_mask]
dup_ids

array([], dtype='<U15')

In [ ]:
INCLUDE_LIST[40]

AttributeError: 'set' object has no attribute 'iloc'

In [ ]:
INCLUDE_LIST=np.asarray(INCLUDE_LIST)


In [ ]:
INCLUDE_LIST

array({'GW190725_174728', 'GW241009_022835', 'GW200202_154313', 'GW231118_005626', 'GW190521_074359', 'GW191129_134029', 'GW230924_124453', 'GW190719_215514', 'GW231108_125142', 'GW231118_071402', 'GW240107_013215', 'GW240703_191355', 'GW241230_233618', 'GW190915_235702', 'GW250119_025138', 'GW190720_000836', 'GW170818_022509', 'GW150914_095045', 'GW190521_030229', 'GW231223_075055', 'GW250118_023225', 'GW240930_035959', 'GW190620_030421', 'GW231231_154016', 'GW231223_202619', 'GW240601_231004', 'GW231004_232346', 'GW200128_022011', 'GW241124_024914', 'GW191230_180458', 'GW230630_125806', 'GW240612_081540', 'GW250118_055802', 'GW240413_022019', 'GW230731_215307', 'GW190727_060333', 'GW241109_115924', 'GW241125_010116', 'GW230628_231200', 'GW241011_233834', 'GW230814_061920', 'GW230708_230935', 'GW191222_033537', 'GW190408_181802', 'GW231129_081745', 'GW240519_012815', 'GW191103_012549', 'GW170809_082821', 'GW230928_215827', 'GW250119_190238', 'GW200311_115853', 'GW230708_053705', 'GW23

In [ ]:
INCLUDE_LIST[40]

IndexError: too many indices for array: array is 0-dimensional, but 1 were indexed